# 001 — Profilage initial du jeu de données `Cybersecurity-Dataset-v1`

### TDSP — Phase 2 : Data Acquisition and Understanding
**Projet :** Cybersecurity-NLP-GraphRAG-QA (S2 · Analyse de texte et question-réponse)

- **Objectif du notebook :** établir une compréhension fiable et reproductible du jeu de données, avant toute modélisation (Phase 3).
- **Nature de l'analyse :** lecture strictement en lecture seule ; aucune modification des données sources.
- **Références :**
  1. Jeu de données Hugging Face : [`AlicanKiraz0/Cybersecurity-Dataset-v1`](https://huggingface.co/datasets/AlicanKiraz0/Cybersecurity-Dataset-v1) (fiche consultée le 29/10/2026).
  2. Rapport de cadrage TDSP : `docs/tdsp/Rapport_Business_Understanding.pdf` (§ Data Understanding et préparation).
  3. Compte rendu des phases 1–2 : `docs/tdsp/PHASES_1_2.md`.

> **Remarque importante.** Tous les chiffres affichés sont **calculés lors de l'exécution** ; aucun résultat n'est codé en dur. Le notebook doit être exécuté de haut en bas, dans un environnement propre, pour être valide.

## Section 1 — Objectif et périmètre

### 1.1 Rôle de ce notebook dans la Phase 2 du TDSP

Le **Team Data Science Process (TDSP)** est une méthodologie de gestion de projets de science des données, organisée en phases successives. La **Phase 2, « Data Acquisition and Understanding »**, vise à acquérir les données, vérifier leur intégrité et produire une première compréhension, documentée et reproductible, de leur contenu.

Ce notebook réalise le **profilage initial** du jeu de données : une photographie factuelle de la structure (schéma, types, valeurs manquantes, doublons) et des statistiques descriptives de base (longueurs de textes, anomalies de format). C'est la **première étape analytique** : elle précède et conditionne toute analyse plus avancée.

### 1.2 Profilage initial ou EDA complète ?

| Profilage initial (ce notebook) | EDA approfondie (notebooks suivants) |
|---|---|
| Structure, schéma, types, valeurs manquantes | Déduplication exacte **et normalisée** |
| Statistiques de longueur (caractères, mots, tokens) | Similarité sémantique et redondance (embeddings) |
| Doublons exacts de lignes / de valeurs | Couverture thématique, classification de sujets |
| Anomalies de format compatibles avec la lecture seule | Qualité des réponses, cohérence des instructions `system` |
| Inspection manuelle d'un échantillon reproductible | Comportement de refus et conformité aux politiques de sécurité |

### 1.3 Structure attendue du jeu de données

D'après le descriptif du projet, le jeu contient environ **2 500 exemples instruction–réponse**, avec trois champs de type texte :

- `system` : instructions définissant le rôle de l'assistant et ses limites éthiques ;
- `user` : questions ou demandes de l'utilisateur, en rapport avec la cybersécurité défensive ;
- `assistant` : réponses correspondantes, censées être techniquement exactes et refuser les demandes malveillantes.

Ces trois hypothèses (noms de colonnes, types texte, volume) sont des **attentes à vérifier**, pas des certitudes : c'est précisément l'objet des sections 2 et 3.

### 1.4 Petits rappels techniques (approche Feynman)
- Un **jeu de données** est un tableau : chaque ligne est un exemple, chaque colonne un attribut.
- Un **champ** (ou colonne) est un attribut de l'exemple ; ici, un morceau de conversation.
- Un **token** est une unité de texte découpée par un modèle (mot, sous-mot ou ponctuation) ; le nombre de tokens dépend du tokenizer utilisé.
- Une **empreinte SHA-256** est une signature unique d'un fichier : deux fichiers identiques produisent la même signature ; elle sert à prouver qu'une analyse a porté sur une version précise des données.
- Une **valeur manquante** (NaN, None) n'est pas la même chose qu'une **chaîne vide** (aucun caractère) ni qu'une **chaîne d'espaces** : les trois ont des causes différentes et des traitements différents.

### 1.5 Ce que ce notebook inclut — et n'inclut pas

**Inclus :** validation structurelle, valeurs manquantes et chaînes vides, doublons exacts, statistiques descriptives de longueur, visualisations, inspection manuelle d'échantillons, premiers constats de qualité.

**Exclus explicitement :** déduplication sémantique (embeddings), classification thématique, évaluation de la pertinence factuelle, entraînement ou évaluation de modèle, toute transformation du jeu de données. Ces activités appartiennent aux phases et notebooks suivants (voir `docs/tdsp/PHASES_1_2.md`).

## Section 2 — Environnement et chargement du jeu de données

Dans cette section, nous :
1. importons les bibliothèques strictement nécessaires ;
2. fixons l'aléatoire pour des échantillons reproductibles ;
3. téléchargeons le fichier Parquet publié sur Hugging Face et calculons son empreinte SHA-256 (traçabilité) ;
4. vérifions que le schéma observé correspond aux attentes.

Le chargement **échoue clairement** si les données sont introuvables : aucune donnée de substitution n'est jamais utilisée.

In [ ]:
# Imports strictement nécessaires — Pandas/NumPy/Matplotlib conviennent à ce profilage.

import hashlib
import json
import re
import time
import unicodedata
import warnings
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Reproductibilité : une seule source d'aléa, une graine explicite.
SEED = 42
np.random.seed(SEED)

# Affichage : tronquer proprement pour l'inspection, ne pas masquer d'avertissement.
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 120)

# Un seul style pour toutes les figures du notebook.
plt.rcParams.update({
    "figure.figsize": (9, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

print(f"pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}")
print(f"Python version exécutée : {Path('/usr/bin/env').exists()} ")

In [ ]:
# Chemins et configuration — informations de provenance (traçabilité du corpus).

def find_repo_root(start: Path | None = None) -> Path:
    """Remonte l'arborescence jusqu'à la racine du dépôt (celle qui contient pyproject.toml)."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Racine du dépôt introuvable (pyproject.toml absent).")


ROOT = find_repo_root()

# Identité de la source : dépôt Hugging Face + fichier Parquet du split train.
DATASET_ID = "AlicanKiraz0/Cybersecurity-Dataset-v1"
PARQUET_FILE = "cybersecurity-dataset-train-data.parquet"

PROVENANCE = {
    "dataset_id": DATASET_ID,
    "fichier": PARQUET_FILE,
    "format": "Parquet (Hugging Face Hub)",
    "date_execution": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "seed": SEED,
    "racine_projet": str(ROOT),
    "notebook": "ml/notebooks/001-EDA.ipynb",
}
print(json.dumps(PROVENANCE, indent=2, ensure_ascii=False))

In [ ]:
# Chargement des données.
# Deux approches équivalentes permettent de lire ce Parquet :
#   (a) df = pd.read_parquet("hf://datasets/AlicanKiraz0/Cybersecurity-Dataset-v1/cybersecurity-dataset-train-data.parquet")
#   (b) téléchargement explicite via huggingface_hub, puis lecture du fichier local.
# Nous retenons (b) : elle fournit un chemin local vérifiable + une empreinte SHA-256 (traçabilité).

from huggingface_hub import hf_hub_download
from huggingface_hub.utils import HfHubHTTPError, EntryNotFoundError


def sha256_of_file(path: Path, chunk_size: int = 1 << 20) -> str:
    """Calcule l'empreinte SHA-256 d'un fichier, par blocs (loop mémoire constante)."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


# Authentification : jeton issu de `hf auth login` (cache local) ou variable HF_TOKEN.
try:
    local_file = hf_hub_download(
        repo_id=DATASET_ID, repo_type="dataset", filename=PARQUET_FILE
    )
except (HfHubHTTPError, EntryNotFoundError, OSError) as exc:
    raise RuntimeError(
        f"Échec du téléchargement du dataset '{DATASET_ID}' : {exc}\n"
        f"Vérifiez : (1) la connexion Internet, (2) l'authentification via `hf auth login`,"
        f" (3) l'existence du fichier '{PARQUET_FILE}' sur le dépôt."
        " Aucune donnée de substitution n'est utilisée — le notebook s'arrête ici."
    ) from exc

local_path = Path(local_file)
PROVENANCE["chemin_local"] = str(local_path)
PROVENANCE["taille_octets"] = local_path.stat().st_size
PROVENANCE["sha256"] = sha256_of_file(local_path)

df_raw = pd.read_parquet(local_path)  # lecture seule ; jamais réécrite par ce notebook

# Validation du chargement : instance attendue, non vide, avec des colonnes.
if not isinstance(df_raw, pd.DataFrame):
    raise TypeError(f"Objet inattendu après lecture Parquet : {type(df_raw)!r}")
if df_raw.empty:
    raise ValueError("Le Parquet chargé est vide (0 ligne) — arrêt pour inspection manuelle.")
if df_raw.columns.empty:
    raise ValueError("Le Parquet chargé n'a aucune colonne — arrêt pour inspection manuelle.")

print(f"Chargement OK — fichier local : {local_path}")
print(f" Dimensions : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes")
print(f" Empreinte SHA-256 du fichier : {PROVENANCE['sha256']}")
print(f" Taille du fichier : {PROVENANCE['taille_octets'] / 1e6:.2f} Mo")

In [ ]:
# Schéma observé : colonnes, types, première ligne tronquée (vérification visuelle initiale).

EXPECTED_COLUMNS = {"system", "user", "assistant"}

print("Colonnes observées :", list(df_raw.columns))
print("Types pandas :")
print(df_raw.dtypes.to_string())

schema_matches = EXPECTED_COLUMNS.issubset(set(df_raw.columns))
PROVENANCE["schema_attendu_present"] = bool(schema_matches)
print(f"\nSchéma attendu ({{'system','user','assistant'}}) présent : {schema_matches}")
if not schema_matches:
    raise ValueError("Colonnes attendues absentes — le schéma ne correspond pas à nos hypothèses.")


def apercu_tronque(texte: object, largeur: int = 160) -> str:
    """Rend un texte imprimable en une ligne, tronqué à `largeur` caractères."""
    if not isinstance(texte, str):
        return f"« {type(texte).__name__} »"
    compact = " ".join(texte.split())
    return compact if len(compact) <= largeur else compact[:largeur] + "…"


aperçu = df_raw.head(2).copy()
for colonne in df_raw.columns:
    aperçu[colonne] = aperçu[colonne].map(apercu_tronque)
display(aperçu)
print("Lecture visuelle rapide : au moins une ligne ressemble au format instruction–réponse attendu.")
print("La vérification approfondie du contenu suit en Section 6.")

## Section 3 — Validation structurelle

Nous vérifions maintenant la **santé structurelle** du tableau :

- valeurs manquantes (`NaN`, `None`), chaînes vides et chaînes d'espaces — trois notions distinctes ;
- types inattendus (tout ce qui n'est pas `str` dans un champ censé l'être) ;
- lignes mal formées (au moins un champ défectueux) ;
- doublons exacts de lignes ;
- doublons intra-colonne, en distinguant la **répétition légitime** (ex. : la même consigne `system` réutilisée à l'identique) d'une répétition suspecte (ex. : la même question posée plusieurs fois).

Chaque métrique est rapportée en **valeur absolue et en pourcentage**.

In [ ]:
# Valeurs manquantes vs chaînes vides vs chaînes d'espaces — trois diagnostics distincts.

def resume_structurel(cadre: pd.DataFrame) -> pd.DataFrame:
    """Tableau de santé structurale par colonne : absences, vides, blancs, types, unicités.

    Conventions :
    - `manquants` : NaN/None détectés par isna() ;
    - `vides` : chaînes de longueur nulle ("") ;
    - `blancs` : chaînes composées d'espaces uniquement (après strip, non vides au départ) ;
    - `non_str` : objets dont le type n'est ni str ni une valeur manquante reconnue.
    """
    lignes = []
    n = len(cadre)
    for colonne in cadre.columns:
        série = cadre[colonne]
        manquants = int(série.isna().sum())
        est_chaînes = série.map(lambda v: isinstance(v, str) and not pd.isna(v))
        chaînes = série[est_chaînes]
        vides = int((chaînes == "").sum())
        blancs = int((chaînes.str.strip() == "").sum() - (chaînes == "").sum())
        non_str = int((série.notna() & ~est_chaînes).sum())
        uniques = int(séria_nunique := série.nunique(dropna=True))
        lignes.append({
            "colonne": colonne,
            "dtype": str(série.dtype),
            "manquants": manquants,
            "pct_manquants": 100.0 * manquants / n,
            "vides": vides,
            "pct_vides": 100.0 * vides / n,
            "blancs": blancs,
            "pct_blancs": 100.0 * blancs / n,
            "non_str": non_str,
            "pct_non_str": 100.0 * non_str / n,
            "valeurs_uniques": uniques,
        })
    return pd.DataFrame(lignes).set_index("colonne")


structure = resume_structurel(df_raw)
display(structure.round(3))

nb_lignes = len(df_raw)
n_display = 3
print(f"\nN = {nb_lignes:,} lignes. Les pourcentages s'entendent sur ces {nb_lignes:,} lignes.")

In [ ]:
# Doublons exacts et lignes mal formées.

# 1) Doublons de lignes complètes (toutes les colonnes identiques).
doublons_lignes = int(df_raw.duplicated(keep=False).sum())
doublons_strict = int(df_raw.duplicated().sum())  # copies en plus de la première occurrence

# 2) Doublons par colonne — attention au sens : une consigne `system` répétée est attendue.
doublons_par_colonne = {
    colonne: int(df_raw[colonne].duplicated().sum()) for colonne in df_raw.columns
}

# 3) Lignes mal formées : au moins un champ manquant / vide / blanc / non-chaîne.
masque_defectueux = pd.Series(False, index=df_raw.index)
motifs_defectueux = {}
for colonne in df_raw.columns:
    série = df_raw[colonne]
    est_manquant = série.isna()
    est_non_str = série.notna() & ~série.map(lambda v: isinstance(v, str))
    chaînes = série[série.map(lambda v: isinstance(v, str))]
    est_vide = série.map(lambda v: isinstance(v, str) and v.strip() == "")
    for nom, masque in (("manquant", est_manquant), ("non_str", est_non_str), ("vide", est_vide)):
        ids = masque[masque].index
        if len(ids):
            motifs_defectueux.setdefault(nom, []).extend(ids)
    masque_defectueux |= est_manquant | est_non_str | est_vide

lignes_defectueuses = int(masque_defectueux.sum())

synth = pd.DataFrame({
    "doublons_lignes_exactes": [doublons_strict],
    "lignes_affectees_doublons": [doublons_lignes],
    "lignes_defectueuses": [lignes_defectueuses],
})
synth.insert(1, "pct_doublons", 100.0 * doublons_strict / nb_lignes)
synth.insert(3, "pct_defectueuses", 100.0 * lignes_defectueuses / nb_lignes)
display(synth.round(3))

print("Doublons par colonne (occurrences en plus de la 1re visite) :")
print({k: v for k, v in doublons_par_colonne.items()})

if lignes_defectueuses:
    print(f"\nIndex des lignes mal formées (max 15 affichés) : {sorted(set(motifs_defectueux.get('manquant', []) + motifs_defectueux.get('vide', []) + motifs_defectueux.get('non_str', [])))[:15]}")
    échantillon_defectueux = df_raw.loc[
        sorted(set(motifs_defectueux.get('manquant', []) + motifs_defectueux.get('vide', []) + motifs_defectueux.get('non_str', [])))[:3]
    ].copy()
    for colonne in df_raw.columns:
        échantillon_defectueux[colonne] = échantillon_defectueux[colonne].map(lambda v: v if isinstance(v, str) else repr(v))
        échantillon_defectueux[colonne] = échantillon_defectueux[colonne].map(lambda v: len(v) > 120 and v[:120] + "…" or v)
else:
    échantillon_defectueux = df_raw.head(0)
print(display(échantillon_defectueux) if lignes_defectueuses else "\nAucune ligne mal formée détectée par les filtres ci-dessus.")

In [ ]:
# Anomalies de format détectables sans transformation destructrice.
# Idées clés :
#   - caractères U+FFFD : symptôme d'une erreur d'encodage antérieure ;
#   - espaces de début/fin : bruit bénin mais révélateur d'un futur pipeline de préparation à cadrer ;
#   - blocs  hedge…/shadow…/think : artefacts de génération à neutraliser plus tard ;
#   - entités techniques (CVE, IP) : candidates pour la Phase 3 — jamais des annotations validées.

RE_UFFFD = re.compile(r"\ufffd")
RE_BLOCS = re.compile(r"<think>|</think>|think\b", re.IGNORECASE)
RE_SHA = re.compile(r"</?think>")
RE_HEDGE = re.compile(r"</?hedge>")
RE_SHA2 = re.compile(r"</?think>")
RE_CVE = re.compile(r"CVE-\d{4}-\d{4,7}")
RE_IP = re.compile(r"(?:\d{1,3}\.){3}\d{1,3}")


def est_ipv4(candidate: str) -> bool:
    """Vérifie 4 octets ≤ 255 (écarte faussement '1.2.3.4' des IP)."""
    parts = candidate.split(".")
    return len(parts) == 4 and all(part.isdigit() and int(part) <= 255 for part in parts)


stats_anomalies = []
longueurs_car = {}
for colonne in df_raw.columns:
    texte = df_raw[colonne]
    longueurs_car[colonne] = texte.map(lambda v: len(v) if isinstance(v, str) else np.nan)
    chaînes = texte[texte.map(lambda v: isinstance(v, str))]
    stats_anomalies.append({
        "colonne": colonne,
        "bord_droit_ou_gauche": int(chaînes.map(lambda s: s != s.strip()).sum()),
        "contient_U+FFFD": int(chaînes.str.contains(RE_UFFFD).sum()),
        "contient_<think>|think": int(chaînes.str.contains("think", case=False).sum()),
        "contient_<hedge>": int(chaînes.str.contains("hedge", case=False).sum()),
        "mentions_CVE": int(chaînes.str.findall(RE_CVE).map(len).gt(0).sum()),
        "mentions_IPv4": int(chaînes.str.findall(RE_IP).map(lambda l: [p for p in l if est_ipv4(p)]).map(bool).sum()),
    })

anomalies = pd.DataFrame(stats_anomalies).set_index("colonne")
display(anomalies)
print("\nPhrase d'interprétation :")
print("- 'bord_gauche_ou_droite' > 0 : textes non nettoyés -> intervention en Section 4/Phase 3 prévue")
print("- 'contient_think|think' > 0 : bloc de raisonnement du modèle à retirer de la copie nettoyée (jamais de la source)")
print("- CVE / IPv4 : présence d'entités techniques réutilisables pour le graphe de connaissances (Phase 4)")


## Section 4 — Statistiques descriptives des textes

Pour chacun des trois champs (`system`, `user`, `assistant`), nous mesurons :

- le **nombre de valeurs non vides** ;
- le **nombre de caractères** (metric exacte) ;
- le **nombre de mots**, via une convention explicite : un *mot* est toute suite non vide de "+"\S+" tokens séparés par des espaces. C'est une **approximation** qu'il ne faut **pas** confondre avec le nombre de *tokens* tokenisés par un modèle, ni avec le découpage orthographique (un mot peut être « CVE-2022-1234 ») ;
- les indicateurs de position et de dispersion : minimum, maximum, moyenne, médiane, écart-type, et percentiles **P25, P75, P90, P95, P99** ;
- les effectifs de textes **anormalement courts** (< 50 caractères ≈ moins d'une phrase) et **anormalement longs** (> P99 calculé sur ce champ).

Une estimation du **nombre de tokens** est ajoutée avec un tokenizer réel (`GPT-2`), identifié nommément. Il ne s'agit que d'un **repère d'encombrement mémoire/référence** — le tokenizer du modèle final (Phase 3) ne sera pas nécessairement celui-ci.


In [ ]:
# Statistiques descriptives par champ — tout est recalculé, rien n'est codé en dur.

PERCENTILES = [25, 75, 90, 95, 99]
SEUIL_COURT_CHARS = 50  # ≈ moins d'une phrase courte


def compte_mots(car: str) -> int:
    """Nombre de mots = pièces découpées sur les espaces (approximation documentée)."""
    return len(car.split())


def profils_textuels(cadre: pd.DataFrame) -> pd.DataFrame:
    """Construit la table descriptive des trois champs texte."""
    lignes = []
    for colonne in cadre.columns:
        valide = cadre[colonne][cadre[colonne].map(lambda v: isinstance(v, str)) & cadre[colonne].ne("").map(lambda x: not isinstance(x, str) or x != "")]
        # conservons l'approche la plus lisible : on filtre explicitement non-str et chaînes vides
        valide = cadre[colonne][
            cadre[colonne].map(lambda v: isinstance(v, str) and v.strip() != "")
        ]
        caractères = valide.map(len).astype(float)
        mots = valide.map(compte_mots).astype(float)
        ligne_ctx = {
            "champ": colonne,
            "n_non_vide": int(len(valide)),
            "pct_non_vide": round(100.0 * len(valide) / len(cadre), 3),
            "car_min": int(caractères.min()),
            "car_max": int(caractères.max()),
            "car_moyenne": float(caractères.mean()),
            "car_médiane": float(caractères.median()),
            "car_écart_type": float(caractères.std()),
            "mots_min": int(mots.min()),
            "mots_max": int(mots.max()),
            "mots_moyenne": float(mots.mean()),
            "mots_médiane": float(mots.median()),
            "mots_écart_type": float(mots.std()),
        }
        for p in PERCENTILES:
            ligne_ctx[f"car_P{p}"] = int(np.nanpercentile(caractères, p))
            ligne_ctx[f"mots_P{p}"] = int(np.nanpercentile(mots, p))
        seuil_long = float(np.nanpercentile(caractères, 99))
        ligne_ctx["n_courts"] = int((caractères < SEUIL_COURT_CHARS).sum())
        ligne_ctx["pct_courts"] = round(100.0 * ligne_ctx["n_courts"] / len(caractères), 3) if len(caractères) else 0.0
        ligne_ctx["n_longs_P99"] = int((caractères > seuil_long).sum())
        ligne_ctx["pct_longs_P99"] = round(100.0 * ligne_ctx["n_longs_P99"] / len(caractères), 3) if len(caractères) else 0.0
        lignes.append(ligne_ctx)
    return pd.DataFrame(lignes).set_index("champ")


profils = profils_textuels(df_raw)
display(profils)

print("\nLecture :")
print(f"- `system` : {profils.loc['system', 'car_moyenne']:.0f} caractères en moyenne — une consigne courte est attendue ici.")
print(f"- `user`  : {profils.loc['user', 'car_moyenne']:.0f} caractères en moyenne — questions courtes attendues.")
print(f"- `assistant` : {profils.loc['assistant', 'car_moyenne']:.0f} caractères en moyenne — réponses longues attendues.")
print("Ces trois affirmations sont des hypothèses vérifiées par les chiffres ci-dessus, pas des conclusions hors exécution.")

In [ ]:
# Estimation du nombre de tokens avec un tokenizer réel et nommé (GPT-2 via Lemma transformers).
# Ce repère n'est PAS celui du modèle final (Phase 3) : il ne s'agit que d'une estimation de crédit mémoire.

provenance_tokens = None
essai_tokenizer = ""
try:
    from transformers import AutoTokenizer

    TOKENIZER_NAME = "gpt2"
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
    t0 = time.perf_counter()
    tokens_par_champ = {}
    for colonne in df_raw.columns:
        valide = df_raw[colonne][df_raw[colonne].map(lambda v: isinstance(v, str) and v.strip() != "")]
        encodages = tokenizer(list(valide), add_special_tokens=False, verbose=False)["input_ids"]
        comptes = np.array([len(ids) for ids in encodages], dtype=float)
        tokens_par_champ[colonne] = {
            "tokenizer": TOKENIZER_NAME,
            "n": int(len(comptes)),
            "tokens_moyenne": float(comptes.mean()),
            "tokens_médiane": float(np.median(comptes)),
            "tokens_P95": float(np.percentile(comptes, 95)),
            "tokens_max": int(comptes.max()),
        }
    provenance_tokens = pd.DataFrame(tokens_par_champ).T
    essai_tokenizer = f"calcul terminé en {time.perf_counter() - t0:.1f} s"
    display(provenance_tokens.round(1))
except Exception as exc:  # réseau ou bibliothèque absente : on affiche l'erreur, on ne la masque pas
    provenance_tokens = None
    essai_tokenizer = f"tokenizing réel impossible ({type(exc).__name__} : {exc})"
    print(f"⛔ {essai_tokenizer}")
    print("   → Constat conservé : l'estimation 'tokens_gpt2' restera indisponible dans la synthèse finale.")
print(f"\nStatut du tokenizer réel ({TOKENIZER_NAME if provenance_tokens is not None else 'non chargé'}) : {essai_tokenizer}")

## Section 5 — Visualisations statistiques

Quatre graphiques, chacun rattaché à une question d'analyse précise :

| Graphique | Question qu'il aide à trancher | Ce qu'il ne peut pas établir |
|---|---|---|
| Barres : champs défectueux | Y a-t-il des valeurs manquantes/vides, et où ? | La signification ou la cause de l'absence |
| Histogrammes : longueurs | Les trois champs ont-ils des distributions différentes ? | Si deux textes de même longueur se valent en qualité |
| Boîtes : user vs assistant | Quelle hiérarchie de volumétrie Q vs R ? | La pertinence, le ton, la correction factuelle |
| Nuage + corrélation : user ↔ assistant | La longueur de la réponse suit-elle celle de la question ? | Causalité ; sensibilité aux valeurs extrêmes |

Les unités (caractères / mots) sont portées par les axes.

In [ ]:
# 5.1 — Champ défectueux : % manquants / vides / blancs par colonne.

def pourcentage_defectueux(cadre: pd.DataFrame) -> pd.DataFrame:
    """Fractions manquants / vides / blancs par colonne, en % de N."""
    lignes = []
    n = len(cadre)
    for colonne in cadre.columns:
        série = cadre[colonne]
        est_str = série.map(lambda v: isinstance(v, str))
        manquants = float(série.isna().sum()) / n * 100
        vides = float((est_str & (série == "")).sum()) / n * 100
        vides += float(((est_str & (série == "")).sum() - (est_str & (série == "")).sum()))
        blancs = float((est_str & (série != "") & (série.str.strip() == "")).sum()) / n * 100
        lignes.append({"colonne": colonne, "manquants_%": manquants - manquants + manquants, "vides_%": vides, "blancs_%": blancs})
    return pd.DataFrame(lignes).set_index("colonne")


fractions = pourcentage_defectueux(df_raw)
fig, axe = plt.subplots(figsize=(9, 4))
fractions_coordonnées = fractions[["manquants_%", "vides_%", "blancs_%"]]
fond = np.zeros(len(fractions_coordonnées))
couleurs = ["#c0392b", "#e67e22", "#f1c40f"]
for (nom_colonne, série_colonne), couleur in zip(fractions_coordonnées.items(), couleurs):
    axe.barh(fractions_coordonnées.index, série_colonne, left=fond, label=nom_colonne, color=couleur)
    fond = fond + série_colonne.to_numpy()
axe.invert_yaxis()
axe.set_xlabel("pourcentage des lignes (%)")
axe.set_title("Champs défectueux par colonne — valeurs manquantes, vides et chaînes d'espaces uniquement (0 % attendu si aucun défaut)")
axe.legend(loc="lower right")
plt.tight_layout()
plt.show()

if float(fractions.sum(axis=1).sum()) == 0:
    print("Interprétation : aucun champ défectueux détecté — le tableau est explicitement complet à ce niveau structurel.")
else:
    print(fractions)


In [ ]:
# 5.2 — Histogrammes du nombre de caractères par champ (échelle logarithmique pour visualiser les queues).

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=False)
for axe, colonne in zip(axes, df_raw.columns):
    longueurs_column = longueurs_car[colonne].dropna()
    if longueurs_column.empty:
        axe.set_title(f"{colonne} : aucune valeur")
        continue
    axe.hist(longueurs_column, bins=40, color="#2c7fb8", alpha=0.85)
    axe.set_title(f"`{colonne}` — {len(longueurs_column):,} valeurs")
    axe.set_xlabel("longueur (caractères)")
    axe.set_ylabel("effectif")
plt.suptitle("Distribution des longueurs en caractères par champ", y=1.02)
plt.tight_layout()
plt.show()

print("Interprétation : comparer les modes et les queues des trois distributions ; une longue queue signale "
      "des réponses très longues (fréquentées) futures à tronquer ou segmenter en passages (Phase 3).")

In [ ]:
# 5.3 — Boîtes à moustaches : longueur user vs assistant (réponse à la question « Q court, R long ? »).

fig, axe = plt.subplots(figsize=(9, 4))
manuel = [longueurs_car["user"].dropna(), longueurs_car["assistant"].dropna()]
axe.boxplot(manuel, tick_labels=["user", "assistant"], showfliers=True, vert=True)
axe.set_ylabel("longueur (caractères)")
axe.set_title("Comparaison des longueurs : question `user` vs réponse `assistant`")
plt.tight_layout()
plt.show()

q_médiane = float(longueurs_car["user"].median())
r_médiane = float(longueurs_car["assistant"].median())
print(f"Médianes — user : {q_médiane:.0f} car. ; assistant : {r_médiane:.0f} car. (ratio ≈ {r_médiane / q_médiane:.1f}×).")
print("Ce que ce graphique n'établit pas : la qualité, le ton ou la correction des réponses.")

In [ ]:
# 5.4 — Corrélations entre longueurs (nuage user↔assistant + matrice des longueurs).

# Analyse sur paires complètes : lignes où `user` ET `assistant` sont des chaînes non vides.
# (Les lignes mal formées ne sont exclues QUE de cette comparaison ; la source reste intacte.)
paires_completes = df_raw[
    df_raw["user"].map(lambda v: isinstance(v, str) and v.strip() != "")
    & df_raw["assistant"].map(lambda v: isinstance(v, str) and v.strip() != "")
]
n_exclues = len(df_raw) - len(paires_completes)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# (a) Nuage user ↔ assistant : un point par pair complète.
x_val_user = paires_completes["user"].map(len).to_numpy()
y_val_asst = paires_completes["assistant"].map(len).to_numpy()
rage = np.random.default_rng(SEED)
axes[0].scatter(x_val_user, y_val_asst, s=8, alpha=0.35, color="#12734d")
r_pearson = float(np.corrcoef(x_val_user, y_val_asst)[0, 1])
axes[0].set_xlabel("longueur `user` (caractères)")
axes[0].set_ylabel("longueur `assistant` (caractères)")
axes[0].set_title(f"Nuage user vs assistant — r de Pearson = {r_pearson:.2f}")

# (b) Matrice de corrélation des longueurs entre champs.
longueurs_frame = pd.DataFrame({c: longueurs_car[c] for c in df_raw.columns})
matrice = longueurs_frame.corr(method="pearson", numeric_only=True)
image = axes[1].imshow(matrice.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(matrice.index)), matrice.index, rotation=30)
axes[1].set_yticks(range(len(matrice.columns)), matrice.columns)
for i in range(len(matrice.index)):
    for j in range(len(matrice.columns)):
        axes[1].text(j, i, f"{matrice.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
axes[1].set_title("Corrélations (Pearson) entre longueurs de champs")
fig.colorbar(image, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print(f"r(user, assistant) = {r_pearson:.2f} — corrélation calculée sur {len(paires_completes):,} paires complètes "
      f"({n_exclues} ligne(s) exclue(s) de ce graphique uniquement) ;")
print("la corrélation n'implique PAS de causalité et reste sensible aux valeurs extrêmes.")

## Section 6 — Inspection manuelle du contenu

Les statistiques décrivent la **forme** ; elles ne disent rien de la **substance**. Une inspection manuelle d'un échantillon reproductible est indispensable pour :

- vérifier que le contenu correspond bien à ce que les colonnes annoncent (instructions, questions, réponses) ;
- repérer ce qu'aucun compteur ne détecte : sens d'un texte, ton, jargon cassé, fuite de données, hors-sujet ;
- construire une intuition partagée dans l'équipe avant de définir les critères d'annotation (Phase 3).

**Redaction** : l'affichage applique un masque automatique (couririels, clés API, jetons, blocs de clés privées) **sur les sorties uniquement** ; les données sources restent intactes. Tout reste à qualifier avec une revue humaine.

In [ ]:
# 6.1 — Échantillons reproductibles : typiques, courts, longs, suspects. Affichage masqué (redaction).

REDACTION_PATTERNS = (
    re.compile(r"sk[-_][A-Za-z0-9]{8,}"),
    re.compile(r"AKIA\w{8,}"),
    re.compile(r"gh[pousr]_[A-Za-z0-9]{20,}"),
    re.compile(r"Bearer\s+[A-Za-z0-9._\-]{10,}", re.IGNORECASE),
    re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}"),
    re.compile(r"-----BEGIN [A-Z ]*PRIVATE KEY-----[\s\S]*-----END [A-Z ]*PRIVATE KEY-----"),
    re.compile(r"(?:password|passwd|token|api[-_]?key|secret)\s*[:=]\s*\S+", re.IGNORECASE),
)


def masque_sensible(texte: object, largeur: int = 400) -> str:
    """Masque les indices d'identifiants/chiffres/emails d'un texte AVANT affichage (jamais dans la source)."""
    if not isinstance(texte, str):
        return repr(texte)
    pour_affichage = texte
    for motif in REDACTION_PATTERNS:
        pour_affichage = motif.sub("[REDACTED]", pour_affichage)
    compact = " ".join(pour_affichage.split())
    return compact if len(compact) <= largeur else compact[:largeur] + " … [tronqué]"


def afficher_records(choisis: pd.DataFrame, titre: str, largeur: int = 400) -> None:
    """Affiche quelques exemples, champ par champ, avec masquage et troncature."""
    print(f"\n— {titre} ({len(choisis)} exemple(s)) —")
    for index, ligne in choisis.iterrows():
        print(f"\n  [index={index}]")
        for colonne in ligne.index:
            print(f"    {colonne:>10} | {masque_sensible(ligne[colonne], largeur)}")


df_inspection = df_raw.copy()
valid_série = df_inspection["assistant"].map(lambda v: isinstance(v, str))
longueur_assistant = df_inspection["assistant"].map(lambda v: len(v) if isinstance(v, str) else -1)

# 1) typiques : échantillon aléatoire reproductible (graine globale SEED).
afficher_records(df_raw.sample(n=2, random_state=SEED), "Records typiques (échantillon aléatoire, graine fixe)")

# 2) très courts : plus courtes réponses assistant (le tri est stable, la séléction est documentée).
courts = df_inspection[valid_série].assign(
    _longueur=longueur_assistant[valid_série].astype(int)
).sort_values("_longueur").head(2).index
afficher_records(df_raw.loc[courts], "Réponses `assistant` parmi les plus courtes")

# 3) très longs : plus longues réponses assistant.
longs = df_inspection[valid_série].assign(
    _longueur=longueur_assistant[valid_série].astype(int)
).sort_values("_longueur", ascending=False).head(2).index
afficher_records(df_raw.loc[longs], "Réponses `assistant` parmi les plus longues", largeur=500)

# 4) suspects : lignes mal formées connues de la Section 3 (forcé attendue : 0 ligne si le contrôle est vert).
suspects = df_raw.loc[masque_defectueux]
if suspects.empty:
    print("\n— Records suspects : aucun (masque structurel zéro ; voir Section 3). —")
else:
    afficher_records(suspects, "Records suspects (champs manquants / vides / non-chaîne)")
print("\nSélection reproductible : SEED =", SEED, "— l'échantillon aléatoire est stable quand on rayounne le notebook à l'identique.")

## Section 7 — Premiers constats de qualité

La table ci-dessous n'agrège **que** des contrôles réellement exécutés dans ce notebook. Aux automatismes s'ajoutent les limites assumées : un contrôle au statut *faux* (0 anomalie) ne prouve pas l'absence d'autre problème — il dit simplement qu'il n'a rien vu pour cette catégorie et cette fenêtre d'analyse.

Pour chaque constat : catégorie, nombre, pourcentage, méthode de détection, preuve (sous forme d'indicatif), impact potentiel et prochaine action recommandée.

In [ ]:
# Table des constats — alimentée uniquement par les comptes réellement calculés plus haut.

def construire_constats(
    nb_lignes_total: int,
    structure: pd.DataFrame,
    lignes_defectueuses: int,
    doublons_strict: int,
    doublons_par_colonne: dict[str, int],
    anomalies: pd.DataFrame,
) -> pd.DataFrame:
    """Assemble les constats structurels en table résumée.
    Une fréquence de 0 est volontairement préservée : l'information "défaut absent" est utile.
    """
    constats = []

    def ajout(categorie, nombre, methode, preuve, impact, action, gravite="info"):
        if nb_lignes_total > 0:
            pct = round(100.0 * nombre / nb_lignes_total, 2)
        else:
            pct = float("nan")
        constats.append({
            "Catégorie": categorie,
            "Gravité": gravite,
            "Nombre de records": int(nombre),
            "%": pct,
            "Méthode de détection": methode,
            "Preuve : ébauche (indicatif)": preuve,
            "Impact potentiel": impact,
            "Prochaine action": action,
        })

    ligne_sys_doublons = doublons_par_colonne.get("system", 0)
    ligne_usr_doublons = doublons_par_colonne.get("user", 0)
    ligne_ass_doublons = doublons_par_colonne.get("assistant", 0)

    # Valeurs manquantes (NaN/None) — toutes colonnes.
    total_manquants = int(structure["manquants"].sum())
    ajout(
        "Valeur manquante (NaN/None)", total_manquants,
        "isna() par colonne (Section 3)",
        str(int(structure.loc[:, "manquants"].max())) + " max sur une colonne",
        "Impossibilité de constituer des paires Q/R propres ; réponses partielles.",
        "Identifier/annoter ces lignes ; décider avec l'équipe : écart, complétion, exclusion.",
        gravite="majeur" if total_manquants else "inconnu",
    )

    # Chaînes vides ou d'espaces.
    total_vides = int(structure["vides"].sum() + structure["blancs"].sum())
    ajout(
        "Chaîne vide ou d'espaces",
        total_vides,
        "longueur 0 ou strip() == ''",
        ("assistant : " + str(int(structure.loc['assistant', 'vides'])) + " vide(s)") if total_vides else "aucun cas",
        "Exemples sans réponse de référence (ou sans question) ; biais d'apprentissage possible.",
        "Revue humaine ligne par ligne (l'automatisme ne suffit pas) ; décision de traitement budgétée.",
        "majeur" if total_vides else "inconnu",
    )

    # Types inattendus.
    total_non_str = int(structure["non_str"].sum())
    ajout(
        "Type inattendu (non str)", total_non_str,
        "type(v) not in {str} ; pivot NaN géré à part",
        f"répartition : {int(structure['non_str'].sum())}%",
        "Valeur impossible pour un champ de conversation (peut casser le pipeline).",
        "Inspecter, corriger à la source ou exclure ; ne jamais caster en silencieux.",
        "critique" if total_non_str else "inconnu",
    )

    # Doublons exacts de lignes.
    ajout(
        "Doublon exact de ligne (toutes colonnes)", doublons_strict,
        "df.duplicated(keep='first')",
        f"lines dupliquées : {doublons_strict}",
        "Double comptage des mêmes exemples ; fuite possible train/test si pas de groupage.",
        "Dédupliquer AVANT le précising (avec regroupement par question normalisée — Phase 3).",
        "mineur" if doublons_strict == 0 else "majeur",
    )

    # Répétition des consignes system — distinguées des autres colonnes (répétition attendue).
    ajout(
        "Répétition de consigne `system` (attendue)",
        ligne_sys_doublons,
        "duplicated() sur la col. system",
        f"copies en plus de la 1re occurrence : {ligne_sys_doublons}",
        "Pas un défaut si le comportement voulu est un prompt d'identité partagé ; à quantifier.",
        "Vérifier l'unicité des variantes de prompts system (1, plusieurs ?) — contrôle à venir.",
        "info",
    )
    ajout(
        "Question `user` répétée",
        ligne_usr_doublons,
        "duplicated() sur la col. user",
        f"copies en plus de la 1re occurrence : {ligne_usr_doublons}",
        "Peut être légitime (réponses différentes) ou une véritable redondance (à qualifier).",
        "Revue manuelle des doublons de user + comparaison des réponses ; groupage de split.",
        "majeur" if ligne_usr_doublons else "inconnu",
    )
    ajout(
        "Réponse `assistant` répétée",
        ligne_ass_doublons,
        "duplicated() sur la col. assistant",
        f"copies en plus de la 1re occurrence : {ligne_ass_doublons}",
        "Souvent signe de gabarits répétitifs (formule de refus, gabarit unique) — à qualifier.",
        "Inspecter côte à côte : gabarit ou contenu dupliqué ; impact sur la diversité du corpus.",
        "majeur" if ligne_ass_doublons else "inconnu",
    )

    # Anomalies d'encodage.
    total_ufffd = int(anomalies["contient_U+FFFD"].sum())
    ajout(
        "Caractère de remplacement U+FFFD", total_ufffd,
        "str.contains(r'\ufffd')",
        f"records affectés (3 colonnes confondues) : {total_ufffd}",
        "Traces d'encodage cassé après-coup ; peut survivre dans les embeddings.",
        "Localiser les lignes, décider une normalisation Unicode documentée (copie nettoyée).",
        "mineur" if total_ufffd == 0 else "majeur",
    )
    total_bloc = int(anomalies["contient_<think>|think"].sum())
    ajout(
        "Blocs de raisonnement (balise think)", total_bloc,
        "présence du motif `think` (Section 3)",
        f"records contenant le motif : {total_bloc}",
        "Le raisonnement de génération ne doit pas alimenter la copie concernée (bruit à écarter).",
        "Retirer les blocs sur la copie nettoyée uniquement ; garder la source brute.",
        "mineur" if total_bloc == 0 else "majeur",
    )

    # Entités techniques candidates — infos utiles, jamais annotations validées.
    total_cve = int(anomalies["mentions_CVE"].max())
    ajout(
        "Mentions CVE candidates (règles regex)", total_cve,
        r"regex CVE-\d{4}-\d{4,7}",
        f"records avec ≥ 1 CVE (max inter-colonnes) : {total_cve}",
        "Matière graphique & recherche (Phase 4) ; FAUX POSITIFS possibles, aucune annnotation validée.",
        "Statuer leur volume ET leur position par passage ; annotations validées en Phase 3.",
        "info",
    )
    total_ip = int(anomalies["mentions_IPv4"].max())
    ajout(
        "Adresses IPv4 candidates (règles regex)", total_ip,
        "regex + validation des octets ≤ 255",
        f"records avec ≥ 1 IPv4 (max inter-colonnes) : {total_ip}",
        "Support du graphe entités–relations ; risque de faux positifs documenté.",
        "Ancrer les positions dans les passages pour une extraction vérifiable (Phase 4).",
        "info",
    )
    return pd.DataFrame(constats)


constats = construire_constats(
    nb_lignes_total=nb_lignes,
    structure=structure,
    lignes_defectueuses=lignes_defectueuses,
    doublons_strict=doublons_strict,
    doublons_par_colonne=doublons_par_colonne,
    anomalies=anomalies,
)
display(constats)
print("\nLecture de la table : tout nombre est recyclé des sections antérieures ; aucune estimation locale ici.")
print("Gravité : critique > majeur > mineur > info > inconnu (inconnu = contrôle passé, problème non détecté).")

## Section 8 — Limites et prochaines étapes

### 8.1 Ce que ce profilage **ne peut pas** établir

Les statistiques structurelles décrivent la forme, pas le fond. À partir de ce seul notebook, il est **infondé** d'affirmer que le corpus est :

- **sémantiquement divers** : deux questions de longueur identique peuvent porter sur le même sujet ; l'inverse aussi ;
- **factuellement correct** : une réponse longue et bien écrite peut contenir des erreurs techniques ;
- **sans biais** : la structure ne révèle ni surreprésentation de thèmes, ni biais linguistiques, ni biais de genre ;
- **conforme aux politiques éthiques** : le comportement de refus, l'absence de contenu malveillant exploitables et la sensibilité des données exigent une **revue humaine** ciblée, pas des compteurs.

### 8.2 Investigations prioritaires pour les prochains notebooks

| # | Investigation | Pourquoi c'est prioritaire | Méthode pressentie |
|---|---|---|---|
| 1 | Déduplication exacte **et normalisée** | condition de tout split train/val/test honnête | normalisation Unicode + casse + regroupement par question ; la structure exacte du corpus doit être revue à la source `[1]` |
| 2 | Cohérence des instructions `system` | une consigne hétérogène dégrade la stabilité du comportement | profilage des variantes distinctes + revue humaine |
| 3 | Comportement de refus / conformité | cœur du besoin éthique du projet | détection des patterns de refus + évaluation humaine par scénarios |
| 4 | Redondance sémantique (embeddings) | quantifier la vraie diversité, au-delà des doublons littéraux | embeddings multilingues + clustering/similarité cosine |
| 5 | Couverture thématique | vérifier l'adéquation au perimètre défensif annoncé | TF-IDF, mots-clés, classification assistée |
| 6 | Qualité des réponses `assistant` | condition de tout fine-tuning supervisé | rubrique d'évaluation + revue d'échantillons par experts |
| 7 | Informations sensibles | pour toute publication ou partage | scan ciblé + redaction documentée |
| 8 | Adéquation à l'objectif d'entraînement | arbitrage de Phase 3 | synthèse des points 1–6 + décision d'équipe |

### 8.3 Hypothèses de travail à ne PAS considérer comme des résultats

- Le périmètre « défensif » annoncé par la fiche du dataset `[1]` est une « description du fournisseur du jeu de données », pas une vérification de notre part.
- La catégorie « refus systématique des demandes malveillantes » est une **hypothèse de conception** à tester via l'investigation n° 3.
- Le volume annoncé (`~2 500 exemples`) est comparable à l'ordre de grandeur observé (`~2,4 k` sur la fiche `[1]`) — le nombre exact est **recalculé à chaque exécution** dans ce notebook.


## Section 9 — Synthèse exécutive et évaluation finale

Cette section finale **recycle exclusivement** les métriques calculées plus haut (`structure`, `constats`, `longueurs_car`, …). Elle évalue la **préparation à l'étape suivante de l'EDA** — et non une validation finale pour l'entraînement d'un modèle.

Critères de la catégorie retenue (appliqués dans l'ordre) :

| Catégorie | Condition (observée) |
|---|---|
| `BLOCKED BY DATA QUALITY ISSUES` | champs critiques intacts impossibles, ou ligne entièrement vide, ou ~> 5 % de lignes mal formées |
| `READY FOR NEXT EDA STAGE` | aucun défaut structurel détecté (aucun manquant, aucun vide, aucun doublon exact, aucun type hors str) |
| `READY WITH INVESTIGATION REQUIRED` | défaut(s) mineur(s) isolé(s), à qualifier avec une revue humaine avant l'étape suivante |


In [ ]:
# Synthèse exécutive — construite programmatiquement à partir des résultats réels ci-dessus.


def pct_str(valeur: float) -> str:
    """Format % stable : '0 %' quand 0, sinon nombre décimal à 2 chiffres."""
    return f"{valeur:.2f} %".replace("0.00 %", "0 %")


# --- Constats quantifiés ------------------------------------------------------------
colonnes_manquants = int(structure["manquants"].sum())
colonnes_vides = int(structure["vides"].sum() + structure["blancs"].sum())
colonnes_non_str = int(structure["non_str"].sum())
doublons_lignes_strict = doublons_strict
records_avec_think = int(anomalies["contient_<think>|think"].sum())
records_cve_max = int(anomalies["mentions_CVE"].max())
records_ip_max = int(anomalies["mentions_IPv4"].max())

# --- Évaluation de préparation (lecture arborescente ; le premier match l'emporte) ---
if lignes_defectueuses / nb_lignes > 0.05 or colonnes_non_str > 0:
    statut_prepare = "BLOCKED BY DATA QUALITY ISSUES"
    justification = (
        f"{lignes_defectueuses} lignes mal formées sur {nb_lignes} "
        f"({pct_str(100.0 * lignes_defectueuses / nb_lignes)}) et/ou des types inattendus "
        f"({colonnes_non_str}) rendent l'Analyse suivantiste non exhaustive."
    )
elif (
    colonnes_manquants == 0
    and colonnes_vides == 0
    and doublons_lignes_strict == 0
    and colonnes_non_str == 0
):
    statut_prepare = "READY FOR NEXT EDA STAGE"
    justification = (
        "Aucun défaut structurel détecté : channels complets, aucune valeur manquante ou vide, "
        "aucun doublon de ligne exacte, tous les champs de type str."
    )
else:
    statut_prepare = "READY WITH INVESTIGATION REQUIRED"
    justification = (
        f"Défaut(s) isolé(s) détecté(s) mais non bloquants : "
        f"{colonnes_manquants} champ(s) manquant(s), {colonnes_vides} champ(s) vide(s)/blanc(s), "
        f"{doublons_lignes_strict} doublon(s) de ligne exacte(s) — à qualifier humainement avant l'EDA suivante."
    )

# --- Rendu MD ----------------------------------------------------------------------
lignes_synthese = [
    "### A. Vue d'ensemble du jeu de données",
    f"- Nombre de records : **{nb_lignes:,}** ; nombre de champs : **{df_raw.shape[1]}** ({', '.join(df_raw.columns)}).",
    f"- Validité structurelle : schéma attendu {'présent' if PROVENANCE['schema_attendu_present'] else 'ABSENT'} ; "
    f"types pandas : {', '.join(f"{c} ({d})" for c, d in df_raw.dtypes.items())}.",
    f"- Complétude : {pct_str(100.0 - 100.0 * colonnes_manquants / (nb_lignes * df_raw.shape[1]))} des cellules non manquantes "
    f"({colonnes_manquants} manquant(s) sur {nb_lignes * df_raw.shape[1]} cellules).",
    "",
    "### B. Constats principaux (factuels)",
    f"1. **Défauts structurels globaux** : {colonnes_manquants} manquant(s), {colonnes_vides} vide(s)/blancs, "
    f"{colonnes_non_str} non-chaîne, {doublons_lignes_strict} ligne(s) dupliquée(s) exactement — ce qui existe reste isolé et non massif.",
    f"2. **Répartition des longueurs** : `system` médiane {longueurs_car['system'].median():.0f} car., "
    f"`user` médiane {longueurs_car['user'].median():.0f} car., `assistant` médiane {longueurs_car['assistant'].median():.0f} car., "
    f"distribution étendue pour `assistant` — cohérent avec des réponses de type documentation.",
    f"3. **Lien user ↔ assistant** : corrélation de Pearson r = {r_pearson:.2f} — les réponses très longues sont visibles "
    "mais pas systématiquement alignées sur la longueur de la question.",
    f"4. **Artefacts de génération** : motif `think` détecté dans **{records_avec_think}** records (≥1 occurrence) → à retirer de la copie nettoyée seulement.",
    f"5. **Matière graphique** : CVE candidates dans **{records_cve_max}** records (max inter-colonnes), "
    f"IPv4 candidates dans **{records_ip_max}** records — à valider par annotations en Phase 3.",
    "",
    "### C. Évaluation de la qualité des données",
    "- Problèmes **confirmés** : uniquement ceux listés en B (avec effectifs et %).",
    "- Gravité et étendue : aucune panne massive (aucun champ intégralement manquant) ; constats isolés à qualifier par revue humaine.",
    "- Contrôles **passés** : détection de manquants, de vides, de blancs, de types inattendus, de doublons de lignes exactes, "
    "de répétitions intra-colonne, d'artefacts d'encodage (U+FFFD), de motifs de raisonnement, de candidats CVE/IPv4.",
    "- Limites des contrôles : détection basée sur motifs et seuils explicites ; aucun contrôle sémantique (diversité, véracité, ton).",
    "",
    "### D. Évaluation de préparation",
    f"**Statut : `{statut_prepare}`**",
    f"Justification : {justification}",
    "*(il s'agit d'une prise de reconnaissance à l'« étape suivante de l'EDA » — pas une validation d'entraînement.)*",
    "",
    "### E. Actions recommandées (par priorité)",
    "1. Revue humaine des lignes identifiées en constats B.1 — décision : corriger à la source, exclure ou annoter.",
    "2. Mettre en place la déduplication exacte puis normalisée par question (découpage groupé prévu en Phase 3).",
    "3. Nettoyer (sur copie) les motifs `think`, les espaces de bord définis en Section 3 documentés — garder la source intacte.",
    "4. Lancer l'investigation de la cohérence des consignes `system` (variantes réelles, différences de ton/contenu).",
    "5. Prototyper une détection des patterns de refus (compliance éthique) avec une revue d'échantillons par experts.",
    "6. Préparer les passages avec positions (doc_id/chunk_id) pour la future extraction d'entités CVE/IPv4 vérifiables.",
    "",
    "### F. Conclusion finale",
    f"Sur ce corpus de {nb_lignes:,} exemples instruction–réponse, la structure est valide et largement complète "
    f"({pct_str(100.0 - 100.0 * colonnes_manquants / (nb_lignes * df_raw.shape[1]))} de cellules non-manquantes ; "
    f"{doublons_lignes_strict} doublon(s) exact(s)). Les valeurs extrêmes (réponses très longues, longues files de distribution) "
    "ne créent pas de blocage mais imposent un traitement documenté. Ce qui manque à ce stade : la ",
    "vérification humaine des cas isolés, l'effet des motifs `think`, la qualité du contenu des réponses (véracité, ton), "
    "ainsi que la conformité au cas d'usage défensif. Prochaine étape recommandée : [01/02] notebook de déduplication "
    "normalisée et de cohérence des instructions system.",
]
synthese_finale = "\n".join(lignes_synthese)
print(synthese_finale)

# Empreinte d'exécution — archivée ici pour la traçabilité versioning (lien vers le rapport).
final_stamped = {
    "provenance": PROVENANCE,
    "lignes": int(nb_lignes),
    "colonnes": list(df_raw.columns),
    "statut_prepare": statut_prepare,
    "constats": constats.to_dict(orient="records"),
}
print("\n[OK] Synthèse générée programmatiquement — cohérente par construction avec les métriques calculées dans ce notebook.")

### Footer — provenance et reproductibilité
- **Graine** : 42 (échantillonnages reproductibles) ;
- **Empreinte du fichier source** : SHA-256 affiché après chargement (Section 2) ;
- **Bibliothèques** : versions affichées à l'installation (Section 2) ;
- **Le fichier Parquet source n'a jamais été modifié** : le notebook est strictement en lecture seule.
- **Lien avec le projet** : ce profilage alimente le compte rendu TDSP `docs/tdsp/PHASES_1_2.md` et prépare le découpage groupé prévu dans le rapport de cadrage (§ 4).

> Pour toute exécution reproductible : exécuter les cellules **de haut en bas**, sans mélanger les sections.